In [17]:
from the_well.data.datasets import TrajectoryData, TrajectoryMetadata
from the_well.benchmark.models import UNetClassic, UNetConvNext
from the_well.data.normalization import ZScoreNormalization
from the_well.data import WellDataset

from the_well.benchmark.metrics.spatial import VRMSE
from typing import Sequence, Dict, Tuple, Optional
from einops import rearrange
from pathlib import Path
import fsspec
import pickle
import torch
import yaml
import sys
import os

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.neural_ops import NeuralOperatorModule

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [18]:
#CONFIG_PATH = f"{PROJECT_ROOT}/configs/turbulent_radiative_layer_2D/WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v0.0.toml"
well_base_path    = f"/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets"
well_dataset_name = 'turbulent_radiative_layer_2D'
stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v1.0_20260323_093126-epoch=188-val_loss=0.0217_portable.ckpt"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_1scales_db2_v1.0_20260323_102317-epoch=178-val_loss=0.0096_portable.ckpt"
#ckpt_base_name    = ckpt_name.split('_v')[0]
#ckpt_path         = f"{PROJECT_ROOT}/experiments/turbulent_radiative_layer_2D/ckpts/{ckpt_name}"

# Load Data

In [19]:
dataset = get_well_dataset(
    path=well_base_path,
    physics=well_dataset_name,
    split='test',
    steps=-1,
)

with fsspec.open(stats_path, mode="r") as f:
    stats = yaml.safe_load(f)

core_field_names          = dataset.core_field_names
core_constant_field_names = dataset.core_constant_field_names

normalizer = ZScoreNormalization(stats, core_field_names, core_constant_field_names)

# Load Baseline Models

In [ ]:
unet_classic  = UNetClassic.from_pretrained("polymathic-ai/UNetClassic-turbulent_radiative_layer_2D")
unet_convnext = UNetConvNext.from_pretrained("polymathic-ai/UNetConvNext-turbulent_radiative_layer_2D")
#tfno        = TFNO.from_pretrained("polymathic-ai/TFNO-turbulent_radiative_layer_2D")
#fno         = NeuralOperatorModule.load_from_checkpoint("/Users/gabrieleaccarino/Workspace/waveletflux/experiments/turbulent_radiative_layer_2D/ckpts/fno_turbulent_radiative_layer_2D_v0.0/last_fp32.pt")

baselines = {
    'UNetClassic'   : unet_classic,
    'UNetConvNext'  : unet_convnext,
    #'TFNO'           : tfno,
    #'TFNO'         : tfno,
}

In [33]:
N_TRAJ        = len(dataset.metadata.n_steps_per_trajectory)   # which trajectories to evaluate 
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0]     # autoregressive steps after the seed window
N_STEPS_INPUT = 4                                              # N_STEPS_INPUT sliding-window width (must match training)

results = {name: [] for name in list(baselines.keys())}

for traj_idx in range(N_TRAJ):
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item          = dataset[traj_idx]
    x_full, _ = get_well_inputs(item, device=device)
    H, W, C = x_full.shape[1:]  # (T, H, W, C)

    # ════════════════════════════════════════════════════════════════════════
    # BASELINES — single deterministic prediction per step
    # ════════════════════════════════════════════════════════════════════════
    for model_name, model in baselines.items():
        print(f"\n  [{model_name}]")
        model.eval()
        model.to(device)

        seed_raw    = x_full[:N_STEPS_INPUT].unsqueeze(0).to(device)
        window_norm = normalizer.normalize_flattened(
            seed_raw, mode='variable')                                       # (1, N_STEPS_INPUT, H, W, C)

        bl_preds = torch.zeros(ROLLOUT_STEPS, H, W, C)

        with torch.no_grad():
            for step in range(ROLLOUT_STEPS):
                gt_idx = N_STEPS_INPUT + step
                if gt_idx >= len(x_full):
                    break

                x_in      = rearrange(window_norm, 'B T H W C -> B (T C) H W')
                yhat      = model(x_in)                                      # (B, C_out, H, W)
                yhat_norm = rearrange(yhat, 'B C H W -> B 1 H W C')

                pred_phys = normalizer.denormalize_flattened(
                    yhat_norm, mode='variable')                              # (1, 1, H, W, C)

                bl_preds[step] = pred_phys.cpu().squeeze(0).squeeze(0)

                window_norm = torch.cat(
                    [window_norm[:, 1:], yhat_norm.detach()], dim=1)

                print(f"    step {step+1:>4d}/{ROLLOUT_STEPS-N_STEPS_INPUT}", end='\r')

        print()
        results[model_name].append(bl_preds)                            # list of (T, H, W, C)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [UNetClassic]
    step   97/97

  [UNetConvNext]
    step   97/97

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [UNetClassic]
    step   97/97

  [UNetConvNext]
    step   97/97

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [UNetClassic]
    step   97/97

  [UNetConvNext]
    step   97/97

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [UNetClassic]
    step   97/97

  [UNetConvNext]
    step   97/97

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [UNetClassic]
    step   97/97


# Save to binary

In [35]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/turbulent_radiative_layer_2D/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    results[model_name] = torch.stack(results[model_name], dim = 0)     # (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
    save_path = save_dir / f"{model_name}_{well_dataset_name}_rollout_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(results[model_name], f)